<a href="https://colab.research.google.com/github/Lalesh2005/computer_vision/blob/main/chapter2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [98]:
# import tensorflow as tf
# import matplotlib.pyplot as plt
# import numpy as np

In [99]:
# IMG_HEIGHT=256
# IMG_WIDTH=256

In [100]:
# def read_and_decode(filename,reshape_dims):
#   img = tf.io.read_file(filename)
#   img = tf.image.decode_jpeg(img,channels=3)
#   img = tf.image.convert_image_dtype(img,tf.float32)
#   return tf.image.resize(img,reshape_dims)

In [101]:
# read_file("/golden_temple.jpeg",[256,256])

In [102]:
# def show_img(filename):
#   img = read_and_decode(filename,[IMG_HEIGHT,IMG_WIDTH])
#   plt.imshow(img.numpy())

In [103]:
# show_img("/golden_temple.jpeg")

In [104]:
# def parse_csvline(csv_row):
#   record_default = ["path","flowers"] # default values for the both path and the flowers.
#   filename,label =tf.io.decode_csv(csv_row,record_default)
#   img = read_and_decode(filename,[IMG_HEIGHT,IMG_WIDTH])
#   return img,label

In [105]:
# dataset = (tf.data.TextLineDataset("/content/drive/MyDrive/Colab Notebooks/flower_photos").map(parse_csvline))

## Simple Linear Model

In [106]:
import tensorflow as tf

In [107]:
IMG_HEIGHT =224
IMG_WIDTH = 224
IMG_CHANNELS = 3
CLASS_NAMES = [item.numpy().decode("utf-8") for item in
               tf.strings.regex_replace(
                 tf.io.gfile.glob("gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/*"),
                 "gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/", "")]
CLASS_NAMES = [item for item in CLASS_NAMES if item.find(".") == -1]
print("These are the available classes:", CLASS_NAMES)

These are the available classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [111]:
def read_and_decode(filename,reshape_dims):
  img = tf.io.read_file(filename)
  img = tf.image.decode_jpeg(img,IMG_CHANNELS)
  img = tf.image.convert_image_dtype(img,tf.float32)
  #label = tf.argmax(tf.math.equal(CLASS_NAMES, label))
  return tf.image.resize(img,reshape_dims)

In [112]:
def parse_csvline(csv_row):
  record_defaults = ["path","flowers"]
  filename,label_string = tf.io.decode_csv(csv_row,record_defaults)
  img = read_and_decode(filename,[IMG_HEIGHT,IMG_WIDTH])
  label = tf.argmax(tf.math.equal(CLASS_NAMES,label_string))
  return img,label

In [113]:
train_dataset = tf.data.TextLineDataset("gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/train_set.csv").map(parse_csvline).batch(10)

eval_dataset=tf.data.TextLineDataset("gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/eval_set.csv").map(parse_csvline).batch(10)

In [119]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(IMG_HEIGHT,IMG_WIDTH,IMG_CHANNELS)),
    tf.keras.layers.Dense(len(CLASS_NAMES))
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [122]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics = ['accuracy']
              )

In [123]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 150528)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │       752,645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 752,645 (2.87 MB)

 Trainable params: 752,645 (2.87 MB)

 Non-trainable params: 0 (0.00 B)

In [124]:
history = model.fit(train_dataset, validation_data=eval_dataset, epochs=10)

Epoch 1/10
    330/Unknown 345s 1s/step - accuracy: 0.2319 - loss: 10.1132

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


330/330 ━━━━━━━━━━━━━━━━━━━━ 382s 1s/step - accuracy: 0.2415 - loss: 10.1058 - val_accuracy: 0.2324 - val_loss: 9.9719
Epoch 2/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 300s 909ms/step - accuracy: 0.2427 - loss: 10.1003 - val_accuracy: 0.2324 - val_loss: 9.9719
Epoch 3/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 220s 667ms/step - accuracy: 0.2427 - loss: 10.1003 - val_accuracy: 0.2324 - val_loss: 9.9719
Epoch 4/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 208s 629ms/step - accuracy: 0.2427 - loss: 10.1003 - val_accuracy: 0.2324 - val_loss: 9.9719
Epoch 5/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 186s 562ms/step - accuracy: 0.2427 - loss: 10.1003 - val_accuracy: 0.2324 - val_loss: 9.9719
Epoch 6/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 173s 525ms/step - accuracy: 0.2427 - loss: 10.1003 - val_accuracy: 0.2324 - val_loss: 9.9719
Epoch 7/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 164s 497ms/step - accuracy: 0.2427 - loss: 10.1003 - val_accuracy: 0.2324 - val_loss: 9.9719
Epoch 8/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 204s 617ms/step - accuracy: 0.2427 - loss: 